In [ ]:
!pip install pyannote.audio

In [3]:
import os
import torch
import glob
import json
from tqdm import tqdm
from pyannote.audio import Pipeline

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("pyannote_hf")

In [6]:
pipeline = Pipeline.from_pretrained("pyannote/voice-activity-detection",
                                    use_auth_token=secret_value_0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline=pipeline.to(device)

Model was trained with pyannote.audio 0.0.1, yours is 3.3.2. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.7.1, yours is 2.6.0+cu124. Bad things might happen unless you revert torch to 1.x.


In [7]:
pipeline

# Parse data path

In [8]:
audios_dir = '/kaggle/input/audio-extracted-data/audio_extract'
all_audio_paths = dict()
for part in sorted(os.listdir(audios_dir)):
    all_audio_paths[part] =  dict()

for data_part in sorted(all_audio_paths.keys()):
    data_part_path = f'{audios_dir}/{data_part}'
    audio_paths = sorted(os.listdir(data_part_path))
    for audio_path in audio_paths:
        audio_id = audio_path.replace('.wav', '')
        audio_path_full = f'{data_part_path}/{audio_path}'
        all_audio_paths[data_part][audio_id] = audio_path_full

In [9]:
all_audio_paths["L21_a"]

{'V001': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V001.wav',
 'V002': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V002.wav',
 'V003': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V003.wav',
 'V005': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V005.wav',
 'V006': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V006.wav',
 'V007': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V007.wav',
 'V008': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V008.wav',
 'V009': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V009.wav',
 'V010': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V010.wav',
 'V011': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V011.wav',
 'V012': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V012.wav',
 'V013': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V013.wav',
 'V014': '/kaggle/input/audio-extracted-data/audio_extract/L21_a/V014.wav',
 'V015': '/k

# Voice Activity Detection

In [ ]:
save_dir_all = './audio_detection'
if not os.path.exists(save_dir_all):
    os.mkdir(save_dir_all)

for key in tqdm(all_audio_paths.keys()):
    save_dir = f'{save_dir_all}/{key}'

    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
        
    audio_paths_dict = all_audio_paths[key]
    audio_ids = sorted(audio_paths_dict.keys())
    for audio_id in tqdm(audio_ids):
        audio_path = audio_paths_dict[audio_id]
        output = pipeline(audio_path)
        
        result = []
        for speech in output.get_timeline().support():
            result.append([speech.start, speech.end])
            
        with open(f'{save_dir}/{audio_id}.json', 'w') as f:
            json.dump(result, f)